# F1 one-lap-ahead pace regression (research only)

Run `python training/pit_strategy/prepare_f1_pilot.py` from the repository root to download the pinned 2021–2024 CSVs. Install `numpy`, `pandas`, and `xgboost` in the notebook kernel before running all cells.

Train CPU XGBoost to predict the next observed green-flag, non-pit lap time from data available at the end of the current lap. The target is a pace delta, reconstructed as current lap time plus predicted delta. Rows must be consecutive laps from the same driver, race, and stint. Compare MAE with previous-lap and rolling-three-lap baselines; split by year (2021-22 train, 2023 validation, 2024 test).

This is public F1 timing data, not AC/ACC data or a pit-stop label. The saved manifest always keeps `deployment_ready` false.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "training/pit_strategy/.local_train/data").is_dir())
DATA = ROOT / "training/pit_strategy/.local_train/data"
OUT = ROOT / "training/pit_strategy/.local_train/pace_regressor"
paths = [DATA / f"session_{year}_V2.csv" for year in range(2021, 2025)]
assert all(path.is_file() for path in paths), "Expected downloaded 2021-2024 F1 CSVs in .local_train/data"
laps = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)

race = ["Year", "GP", "Driver"]
for col in ["Year", "LapNumber", "LapTime", "TyreLife", "PitIn", "PitOut",
            "Position", "Interval_front", "Interval_behind", "TrackStatus"]:
    laps[col] = pd.to_numeric(laps[col], errors="coerce")
laps = (laps.dropna(subset=race + ["LapNumber"])
        .sort_values(race + ["LapNumber"], kind="stable")
        .drop_duplicates(race + ["LapNumber"], keep="last")
        .reset_index(drop=True))
laps["_pit"] = laps[["PitIn", "PitOut"]].fillna(0).gt(0).any(axis=1)
laps["_stint"] = laps.groupby(race, sort=False)["_pit"].cumsum()
group = race + ["_stint"]
compound = laps["Compound"].astype("string").str.upper()
laps["_clean"] = (laps["TrackStatus"].eq(1) & laps["LapTime"].between(60, 300)
                   & laps["TyreLife"].between(1, 70) & compound.isin(["HARD", "MEDIUM", "SOFT"])
                   & ~laps["_pit"])
laps["_usable_time"] = laps["LapTime"].where(laps["_clean"])
laps["_rolling3"] = laps.groupby(group, sort=False)["_usable_time"].transform(
    lambda s: s.rolling(3, min_periods=1).mean())
laps["_pace_delta"] = laps.groupby(group, sort=False)["_usable_time"].diff()
laps["_next_lap"] = laps.groupby(group, sort=False)["LapNumber"].shift(-1)
laps["_next_time"] = laps.groupby(group, sort=False)["LapTime"].shift(-1)
laps["_next_age"] = laps.groupby(group, sort=False)["TyreLife"].shift(-1)
laps["_next_compound"] = laps.groupby(group, sort=False)["Compound"].shift(-1).astype("string").str.upper()
laps["_next_clean"] = laps.groupby(group, sort=False)["_clean"].shift(-1).fillna(False).astype(bool)

laps["compound_hard"] = compound.eq("HARD").fillna(False).astype("int8")
laps["compound_medium"] = compound.eq("MEDIUM").fillna(False).astype("int8")
laps["compound_soft"] = compound.eq("SOFT").fillna(False).astype("int8")
laps["compound_detail"] = pd.to_numeric(
    laps["Compound_Detail"].astype("string").str.extract(r"(\d+)", expand=False), errors="coerce")

features = ["LapTime", "_rolling3", "_pace_delta", "TyreLife", "LapNumber", "Position",
            "Interval_front", "Interval_behind", "compound_detail", "compound_hard",
            "compound_medium", "compound_soft"]
keep = (laps["_clean"] & laps["_next_clean"]
        & laps["_next_lap"].eq(laps["LapNumber"] + 1)
        & laps["_next_age"].eq(laps["TyreLife"] + 1)
        & laps["_next_compound"].eq(compound) & laps["_next_time"].notna())
examples = laps.loc[keep, features + ["Year", "_next_time"]].copy()
examples = examples.rename(columns={"LapTime": "current_lap_time_s", "_rolling3": "rolling_3_lap_s"})
examples["target_delta_s"] = examples["_next_time"] - examples["current_lap_time_s"]
examples = examples.drop(columns="_next_time").reset_index(drop=True)
FEATURES = ["current_lap_time_s", "rolling_3_lap_s", "_pace_delta", "TyreLife", "LapNumber",
            "Position", "Interval_front", "Interval_behind", "compound_detail", "compound_hard",
            "compound_medium", "compound_soft"]
assert not examples.empty and examples["Year"].isin([2021, 2022, 2023, 2024]).all()
print(f"Usable one-lap-ahead examples: {len(examples):,}; yearly counts: {examples['Year'].value_counts().sort_index().to_dict()}")


In [ ]:
import numpy as np
import xgboost as xgb

train = examples[examples["Year"].isin([2021, 2022])]
validation = examples[examples["Year"].eq(2023)]
test = examples[examples["Year"].eq(2024)]
assert len(train) and len(validation) and len(test)
model = xgb.XGBRegressor(
    objective="reg:squarederror", tree_method="hist", device="cpu", n_jobs=1,
    n_estimators=300, max_depth=5, learning_rate=0.04, min_child_weight=20,
    subsample=0.8, colsample_bytree=0.9, reg_lambda=5, random_state=42)
model.fit(train[FEATURES], train["target_delta_s"], verbose=False)

def evaluate(frame):
    actual = frame["current_lap_time_s"].to_numpy() + frame["target_delta_s"].to_numpy()
    predicted = frame["current_lap_time_s"].to_numpy() + model.predict(frame[FEATURES])
    previous_lap = frame["current_lap_time_s"].to_numpy()
    rolling = frame["rolling_3_lap_s"].to_numpy()
    mae = lambda prediction: float(np.mean(np.abs(actual - prediction)))
    return {"n": len(frame), "xgboost_mae_s": mae(predicted),
            "previous_lap_mae_s": mae(previous_lap), "rolling_3_lap_mae_s": mae(rolling)}

metrics = {"validation_2023": evaluate(validation), "test_2024": evaluate(test)}
print(json.dumps(metrics, indent=2))


In [ ]:
import hashlib
import json
import xgboost as xgb

OUT.mkdir(parents=True, exist_ok=True)
model_path = OUT / "pace_delta_xgb.json"
model.save_model(model_path)
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
manifest = {
    "schema_version": 1,
    "artifact_kind": "f1_one_lap_ahead_pace_regression_research",
    "vehicle_category": "f1",
    "deployment_ready": False,
    "blocked_by": ["F1-only timing data; no validated AC/ACC training data",
                   "lap-pace prediction does not label or optimize pit strategy"],
    "target": "next consecutive green, non-pit lap time minus current completed lap time (seconds)",
    "features": FEATURES,
    "split": {"train_years": [2021, 2022], "validation_years": [2023], "test_years": [2024],
              "train_examples": len(train), "validation_examples": len(validation), "test_examples": len(test)},
    "metrics": metrics,
    "xgboost": {"version": xgb.__version__, "objective": "reg:squarederror", "tree_method": "hist", "device": "cpu", "n_jobs": 1},
    "dataset_sha256": {path.name: sha256(path) for path in paths},
    "model_file": model_path.name,
    "model_sha256": sha256(model_path),
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Saved research-only model and manifest under {OUT}")
